In [0]:
--create source table
CREATE OR REPLACE TABLE source_transactions (
    transaction_id INT,
    customer_name STRING,
    amount INT
)
USING DELTA;

INSERT INTO source_transactions VALUES
(101, 'Amit', 5000),
(102, 'Rahul', 3000),
(103, 'Priya', 7000);

SELECT * FROM source_transactions;

In [0]:
--Create Target Table
CREATE OR REPLACE TABLE target_transactions
USING DELTA
AS
SELECT * FROM source_transactions;

SELECT * FROM target_transactions;

Perform changes in source

In [0]:
--Updating Existing Records
UPDATE source_transactions
SET amount = 6000
WHERE transaction_id = 101;

In [0]:
--insert new transactions
INSERT INTO source_transactions
VALUES (104, 'Sneha', 9000);

In [0]:
--Delete transactions
DELETE FROM source_transactions
WHERE transaction_id = 103;

In [0]:
SELECT * FROM source_transactions;

In [0]:
--Simulate CDC: enable Change Data Feed on the source table
--CDF only captures changes made AFTER it is enabled,
--so we apply fresh changes below to generate CDC events
ALTER TABLE source_transactions
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

--Generate new CDC events (update / insert / delete)
UPDATE source_transactions
SET amount = 8500
WHERE transaction_id = 102;

INSERT INTO source_transactions
VALUES (105, 'Vikram', 4000);

DELETE FROM source_transactions
WHERE transaction_id = 101;

--Capture the CDC changes from the change feed
SELECT *
FROM table_changes('source_transactions', 5);

In [0]:
CREATE OR REPLACE TABLE cdc_changes (
    transaction_id INT,
    customer_name STRING,
    amount INT,
    operation STRING
)
USING DELTA;

INSERT INTO cdc_changes VALUES
(101, 'Amit', 6000, 'UPDATE'),
(104, 'Sneha', 9000, 'INSERT'),
(103, 'Priya', 7000, 'DELETE');

SELECT * FROM cdc_changes;

In [0]:
--apply cdc changes using mergeinto
MERGE INTO target_transactions AS tgt

USING cdc_changes AS src

ON tgt.transaction_id = src.transaction_id

WHEN MATCHED AND src.operation = 'DELETE'
THEN DELETE

WHEN MATCHED AND src.operation = 'UPDATE'
THEN UPDATE SET
    tgt.customer_name = src.customer_name,
    tgt.amount = src.amount

WHEN NOT MATCHED AND src.operation = 'INSERT'
THEN INSERT (
    transaction_id,
    customer_name,
    amount
)
VALUES (
    src.transaction_id,
    src.customer_name,
    src.amount
);

In [0]:
SELECT *
FROM target_transactions
ORDER BY transaction_id;